# **Lab 03.2 - Introduction to Deep Q-Network**

##### Copyright by UIT-NC@NT549

## **Some instructions before getting started**:
<div style="font-family: 'Arial'; font-size: 16px; line-height: 1.6; text-align: justify;">

Start the Kernel: At the top right, choose <strong>Select Kernel ➞ Python Environments...</strong>

You can run all code blocks to check: From the menu bar, choose <strong>Run All</strong>.

Complete all code blocks marked with the comment <span style="font-family: monospace; font-weight: bold; color:white; background-color: green;"> ### YOU NEED TO WRITE YOUR CODE BELOW ### </span>

This lab consists of 3 parts:
<ol style="margin-left: 20px;">
  <li><strong>Part 1: DQN on VacuumCleanerEnv</strong> - Custom environment with grid world and visual feedback</li>
  <li><strong>Part 2: DQN on LoadBalancingEnv</strong> - Multi-server task distribution problem</li>
  <li><strong>Part 3: Stable-Baselines3 DQN</strong> - Using production-ready RL library for faster implementation</li>
</ol>
</div>

### Imports and Setup

In [8]:
# ============================================================================
# Imports and runtime stability config
# ============================================================================

import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'

import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
from gymnasium import spaces
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque
import random

# Reduce thread contention in notebook kernels (more stable on shared/limited CPUs).
# Guard set_num_interop_threads so this block is re-runnable in an existing kernel.
torch.set_num_threads(1)
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    # Safe to ignore when interop threads were already configured earlier in this session.
    pass

# Fixed seeds => easier to compare student results across runs
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# Use GPU if available; otherwise CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Quick diagnostic print so students can verify runtime environment
print(f"\n{'='*60}")
print(f"Device: {device}")
print(f"PyTorch Version: {torch.__version__}")
print(f"NumPy Version: {np.__version__}")
print(f"Gymnasium Version: {gym.__version__}")
print(f"Torch Threads: {torch.get_num_threads()}")
print(f"{'='*60}\n")


Device: cpu
PyTorch Version: 2.11.0+cpu
NumPy Version: 2.4.2
Gymnasium Version: 1.2.3
Torch Threads: 1



---

# Shared Components for DQN

The following classes (ReplayBuffer, DQNNetwork, DQNAgent) are reused across all parts of this lab. They are modular and can be adapted for different environments.

### Shared Component 1: Replay Buffer

In [45]:
# ============================================================================
# Shared component: replay buffer
# ============================================================================
# Why this matters:
#+ DQN learns better from randomly sampled past experiences than from strictly
#+ consecutive transitions (which are highly correlated).

class ReplayBuffer:
    """Store and sample experiences for DQN training.

    A transition has shape: (state, action, reward, next_state, done)
    """

    def __init__(self, capacity=10000):
        """Create bounded memory for transitions.

        Args:
            capacity: Maximum transitions kept in memory.
        """
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        """Append one transition into replay memory."""
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        """Randomly sample a mini-batch and convert to tensors on target device.

        Returns:
            states, actions, rewards, next_states, dones
        """
        # Instruction:
        # 1) Use random.sample to get a batch of transitions from self.buffer
        ### YOU NEED TO WRITE YOUR CODE BELOW ###
        batch = random.sample(self.buffer, batch_size)

        # Instruction:
        # 1) Convert each component (state/action/reward/next_state/done) to NumPy arrays.
        # 2) Convert arrays to torch tensors on `device`.
        ### YOU NEED TO WRITE YOUR CODE BELOW ###
        states_np = np.array([t[0] for t in batch], dtype=np.float32)
        actions_np = np.array([t[1] for t in batch], dtype=np.int64)
        rewards_np = np.array([t[2] for t in batch], dtype=np.float32)
        next_states_np = np.array([t[3] for t in batch], dtype=np.float32)
        dones_np = np.array([t[4] for t in batch], dtype=np.float32)

        # 3) Convert numpy arrays to PyTorch tensors and move to device
        ### YOU NEED TO WRITE YOUR CODE BELOW ###
        states = torch.tensor(states_np, dtype=torch.float32).to(device)
        actions = torch.tensor(actions_np, dtype=torch.long).to(device)
        rewards = torch.tensor(rewards_np, dtype=torch.float32).to(device)
        next_states = torch.tensor(next_states_np, dtype=torch.float32).to(device)
        dones = torch.tensor(dones_np, dtype=torch.float32).to(device)

        return states, actions, rewards, next_states, dones

    def __len__(self):
        return len(self.buffer)

print("✓ ReplayBuffer class defined")


✓ ReplayBuffer class defined


### Shared Component 2: DQN Network

In [192]:
# ============================================================================
# Shared component: Q-network
# ============================================================================
# This neural network approximates Q(s, a) for all actions a at once.

class DQNNetwork(nn.Module):
    """Simple MLP-based Deep Q-Network."""
    
    def __init__(self, state_size, action_size, hidden_size=128):
        """Initialize network layers.

        Args:
            state_size: Number of features in state vector.
            action_size: Number of discrete actions.
            hidden_size: Width of hidden layer.
        """
        super(DQNNetwork, self).__init__()
        # Instruction:
        # 1) Define a linear layer (self.fc1) that maps state_size to
        #    a hidden layer of size 64
        # 2) Define a ReLU activation (self.relu)
        # 3) Define a linear layer (self.fc2) that maps the hidden layer
        #    to action_size (output Q-values for each action)
        ### YOU NEED TO WRITE YOUR CODE BELOW ###
        self.fc1 = nn.Linear(state_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, action_size)

        
    
    def forward(self, state):
        """Compute Q-values for each possible action given a state batch."""
        x = self.fc1(state)
        x = self.relu(x)
        q_values = self.fc2(x)
        return q_values

print("✓ DQNNetwork class defined")

✓ DQNNetwork class defined


### Shared Component 3: DQN Agent

In [193]:
# ============================================================================
# Shared component: DQN agent
# ============================================================================

class DQNAgent:
    """Reusable Deep Q-Learning agent for discrete-action environments."""

    def __init__(self, state_size, action_size, learning_rate=1e-3, gamma=0.99):
        """Initialize model, optimizer, and replay memory.

        Args:
            state_size: State feature dimension.
            action_size: Number of valid actions.
            learning_rate: Adam learning rate.
            gamma: Reward discount factor.
        """
        self.state_size = state_size
        self.action_size = action_size
        self.gamma = gamma

        self.epsilon = 1.0
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.995

        # Instruction:
        # 1) Create q_network and target_network as instances of DQNNetwork.
        # 2) Move networks to device (CPU or GPU).
        # 3) Initialize target_network weights to match q_network and set to eval mode.
        ### YOU NEED TO WRITE YOUR CODE BELOW ###
        self.q_network = DQNNetwork(state_size, action_size).to(device)
        self.target_network = DQNNetwork(state_size, action_size).to(device)
        self.target_network.load_state_dict(self.q_network.state_dict())
        self.target_network.eval()

        self.optimizer = optim.Adam(self.q_network.parameters(), lr=learning_rate)
        self.loss_fn = nn.MSELoss()

        self.replay_buffer = ReplayBuffer(capacity=10000)
        self.update_counter = 0

    def select_action(self, state, training=True):
        if training and random.random() < self.epsilon:
            return random.randint(0, self.action_size - 1)

        # Instruction:
        # Convert state to tensor shape [1, state_size], run q_network, return argmax action.
        ### YOU NEED TO WRITE YOUR CODE BELOW ###
        state_array = np.array(state, dtype=np.float32).reshape(1, -1)
        state_tensor = torch.tensor(state_array, dtype=torch.float32).to(device)
        
        with torch.no_grad():
            q_values = self.q_network(state_tensor)
        return q_values.argmax(dim=1).item()

    def store_transition(self, state, action, reward, next_state, done):
        state_array = np.array(state, dtype=np.float32)
        next_state_array = np.array(next_state, dtype=np.float32)
        self.replay_buffer.push(state_array, action, reward, next_state_array, done)

    def train_step(self, batch_size=32):
        if len(self.replay_buffer) < batch_size:
            return

        states, actions, rewards, next_states, dones = self.replay_buffer.sample(batch_size)

        # Instruction:
        # Implement one DQN update step.
        # 1) Compute q_values for chosen actions.
        ### YOU NEED TO WRITE YOUR CODE BELOW ###
        q_values = self.q_network(states).gather(1, actions.unsqueeze(1)).squeeze(1)

        # 2) Compute target_q_values using target network.
        ### YOU NEED TO WRITE YOUR CODE BELOW ###
        with torch.no_grad():
            next_q_values = self.target_network(next_states).max(dim=1)[0]
            target_q_values = rewards + self.gamma * next_q_values * (1 - dones)

        # 3) Compute MSE loss and optimize.
        ### YOU NEED TO WRITE YOUR CODE BELOW ###
        self.optimizer.zero_grad()
        loss = self.loss_fn(q_values, target_q_values)
        loss.backward()
        self.optimizer.step()

        # 4) Sync target network every 100 updates.
        ### YOU NEED TO WRITE YOUR CODE BELOW ###
        self.update_counter += 1
        if self.update_counter % 100 == 0:
            self.target_network.load_state_dict(self.q_network.state_dict())

    def decay_epsilon(self):
        # Decay epsilon after each episode.
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

print("✓ DQNAgent class defined")


✓ DQNAgent class defined


---

# PART 2: DQN on LoadBalancingEnv

A multi-server task distribution environment where the agent learns to distribute tasks optimally.

### 2.1: Define LoadBalancingEnv

In [194]:
# ============================================================================
# Part 2 environment: LoadBalancingEnv
# ============================================================================
# Scenario: tasks arrive over time and must be assigned to one of many servers.
# Objective: minimize total queue length (proxy for waiting time / delay).

class Task:
    """Simple task object with processing demand."""
    def __init__(self, demand=10):
        self.demand = demand

class Server:
    """Server with queue and fixed processing speed per step."""
    def __init__(self, speed=1.0):
        self.speed = speed
        self.queue = 0
    
    def add_task(self, demand):
        """Increase queue by incoming task demand."""
        self.queue += demand
    
    def process(self):
        """Process queued work by speed amount each step."""
        self.queue = max(0, self.queue - self.speed)
    
    def get_utilization(self):
        """Normalized queue value in [0, 1]."""
        return min(1.0, self.queue / 100.0)
    

class LoadBalancingEnv(gym.Env):
    """Environment mô phỏng cân bằng tải nhiều server."""

    def __init__(self, num_servers=3):
        super().__init__()

        self.num_servers = num_servers
        self.servers = [Server(speed=1.0) for _ in range(num_servers)]

        self.action_space = spaces.Discrete(num_servers)

        # SỬA: mở rộng state gồm tải server và tỉ lệ chọn server
        self.observation_space = spaces.Box(
            low=0,
            high=1,
            shape=(num_servers * 2,),
            dtype=np.float32
        )

        self.steps = 0
        self.max_steps = 300

        # SỬA: thêm bộ đếm số lần mỗi server được chọn
        self.selection_counts = np.zeros(num_servers, dtype=np.float32)

        # SỬA: trọng số reward cho queue, độ lệch tải, độ lệch tần suất chọn và lỗi chọn server tải cao
        self.w_queue = 0.3
        self.w_load_balance = 0.6
        self.w_selection_balance = 0.6
        self.w_bad_action = 0.5

    def _get_state(self):
        # state mới gồm utilization và tỉ lệ chọn server
        loads = np.array(
            [server.get_utilization() for server in self.servers],
            dtype=np.float32
        )

        total = np.sum(self.selection_counts)

        if total == 0:
            select_ratio = np.zeros(self.num_servers, dtype=np.float32)
        else:
            select_ratio = self.selection_counts / total

        return np.concatenate([loads, select_ratio]).astype(np.float32)

    def reset(self, seed=None):
        super().reset(seed=seed)

        self.servers = [Server(speed=1.0) for _ in range(self.num_servers)]

        #  reset bộ đếm chọn server ở đầu mỗi episode
        self.selection_counts = np.zeros(self.num_servers, dtype=np.float32)

        self.steps = 0

        return self._get_state(), {}

    def step(self, action):
        self.steps += 1

        # lưu queue trước khi chọn để biết server nào đang nhẹ tải nhất
        queues_before = np.array(
            [server.queue for server in self.servers],
            dtype=np.float32
        )

        task = Task(demand=np.random.randint(5, 15))
        self.servers[action].add_task(task.demand)

        # tăng bộ đếm cho server vừa được chọn
        self.selection_counts[action] += 1

        for server in self.servers:
            server.process()

        queues_after = np.array(
            [server.queue for server in self.servers],
            dtype=np.float32
        )

        # reward mới phạt queue cao, tải lệch, chọn lệch server và chọn server đang tải cao
        queue_penalty = np.mean(queues_after) / 100.0
        load_balance_penalty = np.std(queues_after) / 100.0

        select_ratio = self.selection_counts / np.sum(self.selection_counts)
        ideal_ratio = 1.0 / self.num_servers

        # dùng bình phương để phạt mạnh hơn khi tần suất chọn server bị lệch
        selection_penalty = np.mean((select_ratio - ideal_ratio) ** 2)

        min_queue_before = np.min(queues_before)
        bad_action_penalty = (queues_before[action] - min_queue_before) / 100.0

        reward = -(
            self.w_queue * queue_penalty
            + self.w_load_balance * load_balance_penalty
            + self.w_selection_balance * selection_penalty
            + self.w_bad_action * bad_action_penalty
        )

        terminated = False
        truncated = self.steps >= self.max_steps

        info = {
            "queue_penalty": queue_penalty,
            "load_balance_penalty": load_balance_penalty,
            "selection_penalty": selection_penalty,
            "bad_action_penalty": bad_action_penalty,
            "selected_server": action
        }

        return self._get_state(), reward, terminated, truncated, info

class RoundRobinPolicy:
    def __init__(self, num_servers):
        self.num_servers = num_servers
        self.current_server = 0

    def select_action(self, state=None):
        action = self.current_server
        self.current_server = (self.current_server + 1) % self.num_servers
        return action
print("✓ LoadBalancingEnv class defined")

✓ LoadBalancingEnv class defined


### 2.2: Training DQN on LoadBalancingEnv

In [195]:
# ============================================================================
# Part 2 training loop
# ============================================================================
print("\n" + "="*60)
print("PART 2: DQN ON LOADBALANCINGNV")
print("="*60)

print("\n2.2: Training on LoadBalancingEnv")
print("-" * 60)

env2 = LoadBalancingEnv(num_servers=3)
agent2 = DQNAgent(
    state_size=env2.observation_space.shape[0],
    action_size=env2.action_space.n,
    learning_rate=1e-3
)

num_episodes = 300
batch_size = 32

episode_rewards = []
episode_lengths = []

print(f"\nTraining for {num_episodes} episodes...\n")

for episode in range(num_episodes):
    state, _ = env2.reset()
    episode_reward = 0
    step_count = 0
    done = False

    while not done:
        # Instruction:
        # Complete the DQN interaction loop for LoadBalancingEnv.
        ### YOU NEED TO WRITE YOUR CODE BELOW ###
        action = agent2.select_action(state, training=True)
        
        next_state, reward, terminated, truncated, _ = env2.step(action)
        done = terminated or truncated
        agent2.store_transition(state, action, reward, next_state, done)
        agent2.train_step(batch_size)
        episode_reward += reward
        step_count += 1
        state = next_state

    agent2.decay_epsilon()
    episode_rewards.append(episode_reward)
    episode_lengths.append(step_count)

    if (episode + 1) % 50 == 0:
        avg_reward = np.mean(episode_rewards[-50:])
        print(f"Episode {episode+1:3d}/{num_episodes} | Avg Reward (last 50): {avg_reward:.2f} | Epsilon: {agent2.epsilon:.3f}")

print(f"\nTraining completed!")
print(f"Final average reward (last 50 episodes): {np.mean(episode_rewards[-50:]):.2f}")



PART 2: DQN ON LOADBALANCINGNV

2.2: Training on LoadBalancingEnv
------------------------------------------------------------

Training for 300 episodes...

Episode  50/300 | Avg Reward (last 50): -478.24 | Epsilon: 0.778
Episode 100/300 | Avg Reward (last 50): -532.06 | Epsilon: 0.606
Episode 150/300 | Avg Reward (last 50): -505.86 | Epsilon: 0.471
Episode 200/300 | Avg Reward (last 50): -505.95 | Epsilon: 0.367
Episode 250/300 | Avg Reward (last 50): -507.38 | Epsilon: 0.286
Episode 300/300 | Avg Reward (last 50): -469.29 | Epsilon: 0.222

Training completed!
Final average reward (last 50 episodes): -469.29


### 2.3: Evaluation and Visualization

In [196]:
# ============================================================================
# Part 2 evaluation
# ============================================================================
# Greedy evaluation: check how well the learned routing policy minimizes queues.

print("\n2.3: Evaluation on LoadBalancingEnv")
print("-" * 60)

eval_episodes = 30
eval_rewards = []

print(f"\nEvaluating on {eval_episodes} episodes (greedy policy)...\n")

for episode in range(eval_episodes):
    state, _ = env2.reset()
    episode_reward = 0
    done = False
    
    while not done:
        action = agent2.select_action(state, training=False)
        next_state, reward, terminated, truncated, _ = env2.step(action)
        done = terminated or truncated
        episode_reward += reward
        state = next_state
    
    eval_rewards.append(episode_reward)

print(f"Evaluation Results:")
print(f"  Average Reward: {np.mean(eval_rewards):.2f}")
print(f"  Best Reward: {np.max(eval_rewards):.2f}")
print(f"  Worst Reward: {np.min(eval_rewards):.2f}")
print(f"  Std Dev: {np.std(eval_rewards):.2f}")


2.3: Evaluation on LoadBalancingEnv
------------------------------------------------------------

Evaluating on 30 episodes (greedy policy)...

Evaluation Results:
  Average Reward: -431.67
  Best Reward: -348.92
  Worst Reward: -488.64
  Std Dev: 31.28


### 2.4: Evaluate DQN vs Round Robin with metrics logging

In [197]:
import time
import pandas as pd

def evaluate_policy_with_metrics(policy_name, env, action_func, episodes=100):
    records = []

    for episode in range(episodes):
        state, _ = env.reset()
        done = False

        start_time = time.time()

        total_reward = 0
        total_tasks_created = 0
        total_tasks_accepted = 0
        total_tasks_skipped = 0
        total_tasks_completed = 0

        total_queue_time = 0
        total_load_imbalance = 0
        step_count = 0
        server_selection_counts = np.zeros(env.num_servers)

        while not done:
            action = action_func(state)
            server_selection_counts[action] += 1

            total_tasks_created += 1
            total_tasks_accepted += 1

            next_state, reward, terminated, truncated, _ = env.step(action)

            server_queues = np.array([server.queue for server in env.servers])

            total_reward += reward
            total_queue_time += np.mean(server_queues)
            total_load_imbalance += np.std(server_queues)

            total_tasks_completed += 1 if np.sum(server_queues) > 0 else 0

            step_count += 1
            state = next_state
            done = terminated or truncated

        runtime = time.time() - start_time

        record = {
            "policy": policy_name,
            "episode": episode + 1,
            "total_runtime": runtime,
            "total_tasks_created": total_tasks_created,
            "total_tasks_accepted": total_tasks_accepted,
            "total_tasks_skipped": total_tasks_skipped,
            "total_tasks_completed": total_tasks_completed,
            "total_reward": total_reward,
            "avg_queue_time": total_queue_time / step_count,
            "avg_load_imbalance": total_load_imbalance / step_count,
        }

        for i in range(env.num_servers):
            record[f"server_{i}_selection"] = server_selection_counts[i]

        records.append(record)

    return pd.DataFrame(records)

In [198]:
# Evaluate DQN policy
dqn_env = LoadBalancingEnv(num_servers=3)

dqn_metrics = evaluate_policy_with_metrics(
    policy_name="DQN",
    env=dqn_env,
    action_func=lambda state: agent2.select_action(state, training=False),
    episodes=100
)

# Evaluate Round Robin policy
rr_env = LoadBalancingEnv(num_servers=3)
rr_policy = RoundRobinPolicy(num_servers=rr_env.num_servers)

rr_metrics = evaluate_policy_with_metrics(
    policy_name="Round Robin",
    env=rr_env,
    action_func=lambda state: rr_policy.select_action(state),
    episodes=100
)

# Gộp kết quả 2 policy
comparison_df = pd.concat([dqn_metrics, rr_metrics], ignore_index=True)
comparison_df.to_csv("loadbalancing_policy_sum.csv", index=False)

# Tạo bảng tổng hợp
summary_df = comparison_df.groupby("policy").mean(numeric_only=True).reset_index()

display(summary_df)

,policy,episode,total_runtime,total_tasks_created,total_tasks_accepted,total_tasks_skipped,total_tasks_completed,total_reward,avg_queue_time,avg_load_imbalance,server_0_selection,server_1_selection,server_2_selection
0,DQN,50.5,0.149035,300.0,300.0,0.0,300.0,-427.156860,327.398211,36.737169,105.0,94.0,101.0
1,Round Robin,50.5,0.074340,300.0,300.0,0.0,300.0,-339.982117,326.002322,14.364928,100.0,100.0,100.0


## CONGRATULATIONS TEAM!

Congratulations on completing Lab 03.2 - Introduction to Deep Q-Network!

You have successfully:
- Designed and implemented custom Gymnasium environments (VacuumCleanerEnv, LoadBalancingEnv)
- Trained DQN agents on multiple custom environments
- Learned how to use Stable-Baselines3 for production-ready RL
- Compared manual vs. library-based implementations
- Evaluated agents and visualized training progress

You now have the skills to apply DQN to real-world problems!

References: https://gymnasium.farama.org/ | https://stable-baselines3.readthedocs.io/

## ADDITIONAL INFORMATION

**Author**: M.Sc. Phan Trung Phat - Department of Computer Networks and Communications, UIT

**Contact**: phatpt@uit.edu.vn

**Last Updated**: March, 2026